# 04 — Muestra de unión para la base de elección modal

Genera una muestra local de hasta tres viajes por modo observado para inspeccionar cómo se relacionan `viajes`, `personas`, `hogares`, `vehiculos` y `etapas`. Se mantiene una fila por viaje: persona y hogar se unen por las llaves validadas; vehículos y etapas se resumen para la demostración porque son relaciones uno a muchos.

Las sumas de tiempos/costos y las secuencias de modos de etapa son exploratorias, no reglas de agregación aprobadas. El CSV contiene microdatos e identificadores y permanece bajo `data/interim/`, excluido de Git. No representa una muestra estadística.

In [ ]:
from pathlib import Path
import pandas as pd
from mode_choice.mode_choice_join_preview import build_join_preview

PROJECT_ROOT = next((candidate for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (candidate / 'pyproject.toml').is_file()), None)
assert PROJECT_ROOT is not None, 'Could not locate the project root.'
preview_path, summary = build_join_preview(PROJECT_ROOT, rows_per_mode=3)
preview = pd.read_csv(preview_path, dtype=str, keep_default_na=False)
assert len(preview) == summary['preview_rows'] <= summary['observed_modes'] * 3
assert preview['key_viaje'].nunique() == len(preview)
assert preview['persona__key_persona'].ne('').all()
assert preview['hogar__key_hg'].ne('').all()
summary

In [ ]:
review_columns = [
    'key_viaje', 'modo_principal_agrupado', 'zat_ori', 'zat_des', ' hora_ini ', ' duracion_min ',
    'persona__edad', 'persona__genero', 'persona__licencia_cond_vigente',
    'hogar__ingre_mes_hg', 'hogar__perstotal_hg', 'stage_count', 'stage_mode_sequence',
    'stage_cost_sum_exploratory', 'stage_payment_sum_exploratory',
    'vehicle_records_in_household', 'vehicle_quantity_sum_exploratory', 'vehicle_type_values'
]
display(preview[[column for column in review_columns if column in preview.columns]])
print(f'Archivo de revisión local: {preview_path.relative_to(PROJECT_ROOT)}')

## Recomendación de estructura

Para la futura base plana, conservar una fila por `key_viaje`; unir persona por `key_pers = key_persona` y hogar por `key_hg`. No unir directamente filas de `etapas` o `vehiculos`: eso repetiría el viaje. Mantener las tablas de etapas y vehículos como tablas relacionadas o aprobar explícitamente agregados por viaje/hogar con definiciones documentadas. Elegir variables duplicadas solo después de revisar `outputs/tables/mode_choice_duplicate_field_comparison.csv`. El modo observado es la alternativa elegida; no equivale a contar con atributos de opciones no elegidas.